# Getting started 2/3: create your own encoder, on synthetic neural data

* **Goal**: build an encoder from scratch, fit it to neural data, save it, load it back, and plug it into the package. Every step is explicit, so that you can replace each part (the data, the backbone, the readout) by your own.
* **Why synthetic data**: we make the "neurons" ourselves, from TinyImageNet images and a known ground truth (V1-like Gabor neurons). Fitting takes seconds, and because we know the true responses we can tell exactly how good the fitted encoder is, and what the best possible score is given the noise.
* **Steps**: (1) make synthetic neural data, (2) define a backbone and train it, (3) choose a readout and fit it to the neurons, (4) save and load the encoder, (5) check the encoder contract and compute its Fisher metric J(x), (6) plug in any other PyTorch model.
* **Needs**: TinyImageNet in `data/tiny-imagenet-200` (`bash notebooks/download_tiny_imagenet.sh`, 481 MB). A GPU makes it quick; on a CPU the cell parameters below are reduced automatically.
* **Execution time**: about 1 min on a GPU (backbone training 15 s, each readout fit 5 to 10 s).

## 1. Setup and parameters

In [ ]:
# move to the repository root (the folder containing environment.yml): data/ and runs/ are read from there
import os
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "environment.yml").exists() and root != root.parent:
    root = root.parent
if (root / "environment.yml").exists():
    os.chdir(root)
print("working folder:", Path.cwd())

import copy
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

import multigeom
from multigeom import align, compute_fisher, validate_encoder
from multigeom.data import GaborTeacher, TinyImageNetDataset, make_synthetic_neural_data, noise_ceiling
from multigeom.encoders import NeuralEncoder, SmallCNNBackbone, load_encoder
from multigeom.encoders.fitting import correlation_per_neuron

device = "cuda" if torch.cuda.is_available() else "cpu"

# parameters
N_NEURONS = 50              # neurons to simulate and to predict
NOISE_SIGMA = 0.22          # trial-to-trial noise of the neurons (the paper measures 0.22 in V1)
N_TRAIN, N_TEST = 6000, 2000          # images with neural data, for fitting and for testing
IMAGES_PER_CLASS = 200      # TinyImageNet images per class to train the backbone on (200 classes)
PRETRAIN_EPOCHS = 15
FIT_EPOCHS = 100
SAVE_DIR = Path("runs/getting_started")

if device == "cpu":         # keep the demo to a few minutes on a CPU
    N_TRAIN, N_TEST, IMAGES_PER_CLASS, PRETRAIN_EPOCHS, FIT_EPOCHS = 2000, 1000, 40, 5, 40
print("multigeom", multigeom.__version__, "| device:", device)

## 2. Make synthetic neural data

`GaborTeacher` is a model of V1-like neurons with known receptive fields. Each neuron has a Gabor filter (a position, a size, an orientation, a spatial frequency) and responds with the *energy* of the filter output, the response of a complex cell. It follows the encoder contract, so its responses and its Fisher information are a **ground truth** for everything we build next.

We show it TinyImageNet images, calibrate it so that its responses have unit scale, and add noise as an experiment would: `noisy = clean + sigma * N(0, 1)`. The best correlation any model could reach with the noisy data is the **noise ceiling** (the correlation between the noisy and the clean responses).

In [ ]:
def load_images(dataset):
    """All the images of a TinyImageNet dataset as one tensor [N, 1, 64, 64] in [-1, 1]."""
    return torch.stack([dataset[i][0] if isinstance(dataset[i], tuple) else dataset[i] for i in range(len(dataset))])


t0 = time.time()
neural_images = load_images(TinyImageNetDataset("val"))[: N_TRAIN + N_TEST]         # the images the "experiment" shows
teacher = GaborTeacher(N_NEURONS, seed=0).calibrate(neural_images[:2000])
noisy, clean = make_synthetic_neural_data(teacher, neural_images, noise_sigma=NOISE_SIGMA, seed=1)
ceiling = noise_ceiling(clean, NOISE_SIGMA)

x_train, x_test = neural_images[:N_TRAIN], neural_images[N_TRAIN:]
r_train, r_test = noisy[:N_TRAIN], noisy[N_TRAIN:]               # what the experiment records
c_test = clean[N_TRAIN:]                                         # the ground truth (never seen when fitting)
print(f"{len(neural_images)} images with {N_NEURONS} neurons each, loaded in {time.time() - t0:.0f} s")
print(f"noise ceiling (mean correlation of the noisy with the clean responses): {ceiling.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.3), gridspec_kw={"height_ratios": [1, 1.1]})
for n, ax in enumerate(axes[0]):
    ax.imshow(teacher.filters[n, 0].numpy(), cmap="gray"); ax.set_title(f"neuron {n}", fontsize=9); ax.axis("off")
axes[0, 0].text(-0.1, 0.5, "receptive\nfields", transform=axes[0, 0].transAxes, ha="right", va="center")
for n, ax in enumerate(axes[1]):
    ax.scatter(clean[:300, n], noisy[:300, n], s=4, alpha=0.6)
    ax.set_xlabel("true response", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
axes[1, 0].set_ylabel("recorded", fontsize=8)
plt.show()

## 3. Define the backbone and train it

The **backbone** turns an image into a feature map; the readout will read the neurons out of it. `SmallCNNBackbone` is a small convolutional network (three convolution blocks) that is fast to train. To give it useful features we train it first on a task that has nothing to do with our neurons: classifying the 200 TinyImageNet classes. This is an ordinary PyTorch loop, and you can replace the backbone, the task and the loop by your own: the only requirements are that the backbone maps images `[B, 1, 64, 64]` to a feature map `[B, C, S, S]` and has the attributes `out_channels` (C) and `out_spatial` (S).

(The paper's encoders use a pretrained ImageNet network instead of training a backbone here: see notebook 0.)

In [ ]:
train_set = TinyImageNetDataset("train", images_per_class=IMAGES_PER_CLASS, return_labels=True)
pretrain_images = load_images(train_set).to(device)
pretrain_labels = torch.tensor(train_set.labels, device=device)
print(f"{len(train_set)} images, {len(set(train_set.labels))} classes")

backbone = SmallCNNBackbone(channels=(32, 64, 64)).to(device)
head = nn.Linear(backbone.out_channels, 200).to(device)             # classification head, used for training only
optimizer = torch.optim.AdamW(list(backbone.parameters()) + list(head.parameters()), lr=2e-3, weight_decay=1e-4)

history = {"loss": [], "accuracy": []}
t0 = time.time()
for epoch in range(PRETRAIN_EPOCHS):
    backbone.train()
    order = torch.randperm(len(pretrain_images), device=device)
    total, correct = 0.0, 0
    for i in range(0, len(order), 128):
        idx = order[i: i + 128]
        logits = head(backbone(pretrain_images[idx]).mean(dim=(-2, -1)))      # average the feature map, then classify
        loss = nn.functional.cross_entropy(logits, pretrain_labels[idx])
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * len(idx); correct += (logits.argmax(1) == pretrain_labels[idx]).sum().item()
    history["loss"].append(total / len(order)); history["accuracy"].append(correct / len(order))
print(f"backbone trained in {time.time() - t0:.0f} s: classification accuracy {history['accuracy'][-1]:.2f} on the training images "
      f"(chance is 0.005)")
backbone = backbone.cpu()
del pretrain_images

fig, ax = plt.subplots(1, 2, figsize=(7, 2.4))
ax[0].plot(history["loss"]); ax[0].set_xlabel("epoch"); ax[0].set_title("classification loss")
ax[1].plot(history["accuracy"]); ax[1].set_xlabel("epoch"); ax[1].set_title("accuracy")
plt.tight_layout(); plt.show()

## 4. Build the encoder and fit its readout

`NeuralEncoder(backbone, n_neurons, readout=...)` puts a readout on the backbone. The **readout** is the part fitted to neural data. The package has three, chosen by name:

| `readout=` | each neuron reads the feature map with... | parameters |
|---|---|---|
| `"linear"` | the average of the map over space, then a linear map (no spatial selectivity) | N (C + 1) |
| `"gaussian"` | a Gaussian window with a learned centre and width (the readout of the paper) | N (C + 4) |
| `"factorized"` | a free spatial window and feature weights | N (S² + C + 1) |

`encoder.fit_readout(images, responses)` freezes the backbone, computes its features once, fits the readout (AdamW, mean squared error, 5% of the images kept for validation), and keeps the best epoch. We fit each readout on the same backbone and score it on the held-out images, **against the true responses**, which only synthetic data allow. For comparison, the last row uses a backbone that was never trained.

In [ ]:
def score(encoder):
    pred = encoder.predict(x_test, device=device)
    return correlation_per_neuron(pred, c_test).mean(), correlation_per_neuron(pred, r_test).mean()


fits = {}
for name in ("linear", "gaussian", "factorized"):
    enc = NeuralEncoder(copy.deepcopy(backbone), N_NEURONS, readout=name)
    t0 = time.time()
    report = enc.fit_readout(x_train, r_train, epochs=FIT_EPOCHS, lr=3e-3, batch_size=128, device=device, verbose=False)
    fits[name] = (enc, report, score(enc), time.time() - t0)
untrained = NeuralEncoder(SmallCNNBackbone(channels=(32, 64, 64)), N_NEURONS, readout="gaussian")
untrained.fit_readout(x_train, r_train, epochs=FIT_EPOCHS, lr=3e-3, batch_size=128, device=device, verbose=False)
fits["gaussian, untrained backbone"] = (untrained, None, score(untrained), None)

print(f"{'readout':30s} {'validation':>10s} {'test vs truth':>14s} {'test vs recorded':>17s}")
for name, (enc, report, (vs_clean, vs_noisy), seconds) in fits.items():
    val = f"{report.best_val_corr:.3f}" if report else "-"
    print(f"{name:30s} {val:>10s} {vs_clean:14.3f} {vs_noisy:17.3f}")
print(f"{'noise ceiling':30s} {'':>10s} {'1.000':>14s} {ceiling.mean():17.3f}")

Reading the table: *test vs truth* is the correlation with the noise-free responses, which cannot be measured with real recordings; *test vs recorded* is what an experiment would report, and its ceiling is the noise ceiling. The Gaussian window beats the linear readout because these neurons look at one place in the image, and a trained backbone beats an untrained one. The score is below the ceiling because a small convolutional network does not contain the exact Gabor features of the neurons; a wider backbone, other training data or a longer fit can close part of the gap, and swapping them is the point of this notebook.

We keep the readout with the best **validation** correlation (the choice must not use the test images):

In [ ]:
best = max((n for n in fits if fits[n][1] is not None), key=lambda n: fits[n][1].best_val_corr)
encoder = fits[best][0]
print("selected readout:", best)
print(encoder.summary())
print(fits[best][1])

pred = encoder.predict(x_test, device=device)
corr = correlation_per_neuron(pred, c_test)
fig, axes = plt.subplots(1, 4, figsize=(12, 2.8))
for ax, n in zip(axes[:3], np.argsort(corr)[[-1, len(corr) // 2, 0]]):
    ax.scatter(c_test[:, n], pred[:, n], s=3, alpha=0.4); ax.set_title(f"neuron {n}: r = {corr[n]:.2f}", fontsize=10)
    ax.set_xlabel("true response"); ax.set_ylabel("predicted")
axes[3].hist(corr, bins=15, alpha=0.8, label="fitted encoder")
axes[3].set_xlabel("correlation with the true responses"); axes[3].set_ylabel("neurons")
plt.tight_layout(); plt.show()

## 5. Save and load the encoder

`encoder.save(path, **metadata)` writes one file with the weights, the **architecture** (which backbone and readout, with which arguments) and any metadata you give. `NeuralEncoder.load(path)` rebuilds the encoder from it, with no other code needed, and `load_encoder(path)` does the same. This works because the backbone and the readout are *registered* by name: the readouts and `SmallCNNBackbone` are, and `register_backbone("name")` registers one of your own (see the last section). `encoder.load_checkpoint(path)` loads only the weights into an encoder that you built yourself.

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
path = encoder.save(SAVE_DIR / "encoder.pt", area="synthetic", noise_sigma=NOISE_SIGMA, test_corr_vs_truth=float(corr.mean()))
torch.save({"state_dict": teacher.state_dict(), "n_neurons": N_NEURONS, "seed": 0}, SAVE_DIR / "teacher.pt")   # the ground truth, for notebook 3
print("saved:", path, f"({path.stat().st_size / 1e6:.1f} MB)")
print("architecture:", encoder.config)

reloaded = NeuralEncoder.load(path)
same = torch.allclose(reloaded.predict(x_test[:50]), encoder.predict(x_test[:50]), atol=1e-4)    # up to GPU/CPU rounding
print("the reloaded encoder gives the same predictions:", same)
print("metadata:", {k: v for k, v in torch.load(path, weights_only=False)["metadata"].items()})
assert same and torch.allclose(load_encoder(path).predict(x_test[:5]), encoder.predict(x_test[:5]), atol=1e-4)

## 6. Use it in the package

The encoder follows the *encoder contract*, so `validate_encoder`, `compute_fisher` and the rest of the pipeline accept it. We compute the Fisher information J(x) of the fitted encoder and of the **ground-truth teacher** at the same image: the leading eigenvectors are the image changes that move the responses the most.

In [ ]:
report = validate_encoder(encoder, sigma=NOISE_SIGMA)
print("contract satisfied:", report.n_neurons, "neurons | warnings:", report.warnings or "none")

dev = torch.device(device)
x = x_test[:1]
j_fit = compute_fisher(image=x, encoder=encoder.to(dev), sigma=NOISE_SIGMA, top_k=20, device=dev)
j_true = compute_fisher(image=x, encoder=teacher.to(dev), sigma=NOISE_SIGMA, top_k=20, device=dev)

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for row, (label, j) in enumerate([("fitted encoder", j_fit), ("ground truth", j_true)]):
    axes[row, 0].imshow(x[0, 0], cmap="gray", vmin=-1, vmax=1); axes[row, 0].set_title(f"{label}", fontsize=10)
    for k in range(4):
        v = j.eigenvectors[k].reshape(64, 64).cpu(); m = v.abs().quantile(0.99)
        axes[row, k + 1].imshow(v, cmap="RdBu_r", vmin=-m, vmax=m); axes[row, k + 1].set_title(f"eigenvector {k + 1}", fontsize=10)
for ax in axes.ravel(): ax.axis("off")
plt.show()

# how well do the leading eigenvectors of the fitted encoder match the ground truth, over 20 images?
scores = {k: [] for k in (1, 5, 10)}
for i in range(20):
    xi = x_test[i: i + 1]
    a = compute_fisher(image=xi, encoder=encoder, sigma=NOISE_SIGMA, top_k=10, device=dev)
    b = compute_fisher(image=xi, encoder=teacher, sigma=NOISE_SIGMA, top_k=10, device=dev)
    for k in scores:
        scores[k].append(align(a.eigenvectors.cpu(), b.eigenvectors.cpu(), k))
print("subspace alignment between the fitted encoder and the ground truth (1 = same subspace, chance = k / 4096):")
for k, v in scores.items():
    print(f"  top {k:2d} eigenvectors: {np.mean(v):.3f}   (chance {k / 4096:.4f})")

A good fit of the *responses* does not guarantee that the *geometry* is recovered: encoders that predict the same neurons about equally well can have different Fisher metrics. This sensitivity to modelling choices is what the paper studies, and why it defines a multi-scale metric G(x) that is more robust (notebook 3).

## 7. Plug in any other PyTorch model

You do not need `NeuralEncoder` to use the tools: any `nn.Module` that maps images `[B, 1, 64, 64]` in `[-1, 1]` to responses `[B, N]`, is deterministic in evaluation mode and differentiable with respect to its input, works. `validate_encoder` tells you the rule a model breaks and how to fix it.

In [ ]:
class MyModel(nn.Module):
    """Any model of your own that is already fitted: here a tiny random convolutional network."""

    def __init__(self, n_neurons=20):
        super().__init__()
        torch.manual_seed(0)
        self.n_neurons = n_neurons
        self.conv = nn.Conv2d(1, 4, 8, stride=8)
        self.linear = nn.Linear(4 * 8 * 8, n_neurons)

    def forward(self, x):
        return torch.tanh(self.linear(torch.relu(self.conv(x)).flatten(1)))


my_model = MyModel()
print(validate_encoder(my_model, sigma=NOISE_SIGMA))
fisher = compute_fisher(image=x.cpu(), encoder=my_model, sigma=NOISE_SIGMA, top_k=5, device=torch.device("cpu"))
print("leading eigenvalues of J(x):", [f"{v:.3g}" for v in fisher.eigenvalues.tolist()])

class Broken(MyModel):                                      # a rule violation, and the message that names it
    def forward(self, x):
        return super().forward(x).detach()                  # cuts the gradient: J(x) needs it


try:
    validate_encoder(Broken())
except multigeom.EncoderContractError as err:
    print("rejected:", err)

## Recap

| To do | Call |
|---|---|
| make synthetic neurons with a ground truth | `GaborTeacher`, `make_synthetic_neural_data`, `noise_ceiling` |
| define a backbone | any `nn.Module` with `out_channels` and `out_spatial`, e.g. `SmallCNNBackbone` |
| build the encoder | `NeuralEncoder(backbone, n_neurons, readout="gaussian")` |
| fit it to neural data | `encoder.fit_readout(images, responses)` |
| predict neural responses | `encoder.predict(images)` |
| save and load | `encoder.save(path)`, `NeuralEncoder.load(path)`, `load_encoder(path)` |
| register your own backbone or readout | `@register_backbone("name")`, `@register_readout("name")` |
| check the contract | `multigeom.validate_encoder(model)` |

**Next**: [notebook 2](2_getting_started_diffusion_and_pipeline.ipynb) trains the diffusion model for this encoder and runs the pipeline: Fisher information J(x), multi-scale metric G(x), and the figures.